[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/36_explainable_ai/02_local_explanations_lime_shap.ipynb)

# Bài 02 — Giải thích cục bộ: LIME, SHAP và counterfactual

**🎯 Mục tiêu**
- Trả lời "vì sao **khách hàng này** bị dự đoán rời bỏ?".
- Cài **LIME** from scratch; thấy độ nhạy của nó với số mẫu và độ rộng kernel.
- **Giá trị Shapley**: định nghĩa từ lý thuyết trò chơi, các tính chất, cài **chính xác** from scratch; đối chiếu thư viện `shap`; trường hợp mô hình tuyến tính.
- Từ cục bộ đến toàn cục: tổng hợp SHAP của nhiều khách hàng.
- **Counterfactual**: thay đổi tối thiểu nào làm đổi dự đoán — giải thích có thể **hành động**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/36_explainable_ai

In [ ]:
import itertools
import logging
import time
import warnings
from math import factorial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("shap").setLevel(logging.ERROR)
raw = make_customer_churn(20000, seed=0)
X = pd.DataFrame({
    "tenure": raw.tenure_months, "monthly": raw.monthly_charges, "total": raw.total_charges,
    "support_calls": raw.support_calls, "month_to_month": (raw.contract == "month-to-month").astype(int),
    "two_year": (raw.contract == "two_year").astype(int), "fiber": (raw.internet_service == "fiber").astype(int),
    "ewallet": (raw.payment_method == "e-wallet").astype(int), "senior": raw.senior.astype(int),
    "n_products": raw.n_products,
})
y = raw.churn.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(X_train, y_train)
f = model.decision_function                                            # giải thích trên thang log-odds (cộng tính)
FEATURES = list(X.columns)

customer = X_test.iloc[[0]]
print(customer.T.rename(columns={customer.index[0]: "khách hàng"}))
print(f"\nxác suất rời bỏ dự đoán = {model.predict_proba(customer)[0, 1]:.1%} | log-odds = {f(customer)[0]:.3f} "
      f"| log-odds trung bình trên dữ liệu = {f(X_train).mean():.3f}")

## 1. LIME — xấp xỉ tuyến tính cục bộ

LIME (Ribeiro et al., 2016): quanh điểm cần giải thích, tạo nhiều mẫu **nhiễu loạn**, hỏi mô hình dự đoán, rồi fit một **mô hình tuyến tính có trọng số** (mẫu càng gần điểm gốc càng nặng). Hệ số tuyến tính là lời giải thích.

Với dữ liệu bảng, một cách nhiễu loạn tự nhiên: mỗi đặc trưng **giữ giá trị của khách hàng** hoặc **lấy giá trị của một khách hàng ngẫu nhiên** trong dữ liệu. Biểu diễn dễ hiểu $z \in \{0, 1\}^d$ (1 = giữ). Trọng số theo khoảng cách $\pi(z) = \exp(-D(z)^2/\text{width}^2)$, $D$ = căn tỉ lệ đặc trưng bị thay.

In [ ]:
def lime_explain(f, x, background, n_samples=2000, width=0.75, seed=0):
    rng = np.random.default_rng(seed)
    donors = background.sample(n_samples, replace=True, random_state=int(rng.integers(1_000_000_000))).reset_index(drop=True)
    keep = rng.random((n_samples, x.shape[1])) < 0.5                   # z: giữ giá trị của x hay không
    perturbed = donors.copy()
    for j, col in enumerate(x.columns):
        perturbed.loc[keep[:, j], col] = x[col].values[0]
    distance = np.sqrt((~keep).sum(1) / x.shape[1])
    weights = np.exp(-distance**2 / width**2)
    local_model = Ridge(alpha=1.0).fit(keep.astype(float), f(perturbed), sample_weight=weights)
    return pd.Series(local_model.coef_, index=x.columns)


lime_main = lime_explain(f, customer, X_train)
print("LIME (2 000 mẫu):\n", lime_main.round(3).sort_values(key=abs, ascending=False))

Hệ số dương = giá trị của khách hàng ở đặc trưng đó **đẩy** log-odds rời bỏ lên (so với khi thay bằng giá trị ngẫu nhiên). Nhưng LIME có nhiều "nút vặn": số mẫu, độ rộng kernel, cách nhiễu loạn, mô hình địa phương. Thử độ ổn định:

In [ ]:
for n in [100, 2000]:
    runs = pd.DataFrame([lime_explain(f, customer, X_train, n_samples=n, seed=s) for s in range(10)])
    top = runs.abs().idxmax(axis=1).value_counts().to_dict()
    print(f"{n:5d} mẫu: độ lệch chuẩn hệ số giữa 10 lần chạy (trung bình các đặc trưng) = {runs.std().mean():.3f} | "
          f"đặc trưng quan trọng nhất qua 10 lần: {top}")
widths = pd.DataFrame({w: lime_explain(f, customer, X_train, n_samples=5000, width=w) for w in [0.25, 0.75, 5.0]})
print("\nhệ số theo độ rộng kernel:\n", widths.round(3))

Với ít mẫu, chạy lại cho lời giải thích **khác** — thậm chí đổi đặc trưng quan trọng nhất. Độ rộng kernel cũng làm đổi độ lớn hệ số. Lời giải thích phụ thuộc vào các lựa chọn tuỳ ý của người giải thích: luôn cố định seed, dùng đủ mẫu, và kiểm tra độ ổn định trước khi tin.

## 2. Giá trị Shapley

Lý thuyết trò chơi hợp tác (Shapley, 1953): các đặc trưng là "người chơi" cùng tạo ra "phần thưởng" $f(x) - \mathbb E[f(X)]$; chia phần thưởng công bằng bằng cách lấy **trung bình đóng góp biên** của mỗi người chơi qua **mọi thứ tự** tham gia:

$$\phi_j = \sum_{S\subseteq F\setminus\{j\}}\frac{|S|!\,(|F| - |S| - 1)!}{|F|!}\left[v(S\cup\{j\}) - v(S)\right]$$

Hàm giá trị $v(S)$ = dự đoán trung bình khi **biết** các đặc trưng trong $S$ (lấy từ $x$) và các đặc trưng còn lại lấy từ dữ liệu nền (**interventional**): $v(S) = \mathbb E_{X'}[f(x_S, X'_{\bar S})]$.

Giá trị Shapley là cách chia **duy nhất** thoả bốn tính chất: **hiệu quả** ($\sum_j\phi_j = f(x) - \mathbb E[f]$), **đối xứng**, **dummy** (đặc trưng không ảnh hưởng → 0), **cộng tính**. SHAP (Lundberg & Lee, 2017) là khung dùng giá trị Shapley để giải thích dự đoán.

Với 10 đặc trưng có $2^{10} = 1\,024$ tập con — tính chính xác được:

In [ ]:
def exact_shapley(f, x, background):
    d = x.shape[1]
    cols = list(x.columns)
    value = {}
    for size in range(d + 1):
        for S in itertools.combinations(range(d), size):
            Z = background.copy()
            for j in S:
                Z[cols[j]] = x[cols[j]].values[0]
            value[S] = f(Z).mean()
    phi = np.zeros(d)
    for j in range(d):
        for S, v_S in value.items():
            if j not in S:
                weight = factorial(len(S)) * factorial(d - len(S) - 1) / factorial(d)
                phi[j] += weight * (value[tuple(sorted(S + (j,)))] - v_S)
    return pd.Series(phi, index=cols), value[()]


background = X_train.sample(200, random_state=0)
t0 = time.perf_counter()
phi, base = exact_shapley(f, customer, background)
print(f"{time.perf_counter() - t0:.1f}s")
print(pd.DataFrame({"Shapley": phi, "LIME": lime_main}).round(3).sort_values("Shapley", key=abs, ascending=False))
print(f"\nhiệu quả: Σφ = {phi.sum():.4f} | f(x) − E[f] = {f(customer)[0] - base:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = phi.abs().sort_values().index
ax.barh(order, phi[order], color=["C3" if v > 0 else "C0" for v in phi[order]])
ax.set_xlabel("đóng góp vào log-odds rời bỏ (Shapley)")
ax.set_title(f"E[f] = {base:.2f} → f(x) = {f(customer)[0]:.2f}")
for k, feat in enumerate(order):
    ax.text(0, k, f" {feat} = {customer[feat].values[0]}", va="center", fontsize=8, ha="left" if phi[feat] < 0 else "right")
plt.tight_layout()
plt.show()

Tổng các đóng góp **đúng bằng** chênh lệch giữa dự đoán của khách hàng và dự đoán trung bình — lời giải thích "khép kín" (LIME không có đảm bảo này). Ở đây LIME (với cách nhiễu loạn "giữ hoặc thay bằng mẫu nền") cho kết quả gần với Shapley: LIME với kernel và trọng số phù hợp chính là **KernelSHAP**.

### Đối chiếu với thư viện `shap`

In [ ]:
try:
    import shap

    exact_lib = shap.explainers.Exact(f, shap.maskers.Independent(background, max_samples=len(background)))(customer)
    kernel_lib = shap.KernelExplainer(f, background).shap_values(customer, nsamples=2048, silent=True)
    print(f"sai lệch với shap.explainers.Exact: {np.abs(exact_lib.values[0] - phi.values).max():.1e}")
    print(f"sai lệch với shap.KernelExplainer : {np.abs(np.asarray(kernel_lib)[0] - phi.values).max():.1e}")
except ImportError:
    print("Chưa cài shap (pip install -e \".[xai]\") — bỏ qua phần đối chiếu.")

Cài đặt từ định nghĩa khớp thư viện tới sai số máy. Với nhiều đặc trưng hơn, không thể duyệt $2^d$ tập con: KernelSHAP lấy mẫu tập con; **TreeSHAP** (Lundberg et al., 2020) tính chính xác trong thời gian đa thức cho mô hình cây; DeepSHAP/Integrated Gradients cho mạng nơ-ron. Khi dùng, hãy kiểm tra tính chất hiệu quả (Σφ = f(x) − E[f]) như một phép thử nhanh.

### Mô hình tuyến tính: Shapley có dạng đóng

Với $f(x) = \beta_0 + \sum_j\beta_jx_j$ và các đặc trưng được can thiệp độc lập: $\phi_j = \beta_j\,(x_j - \mathbb E[x_j])$ — "hệ số × độ lệch so với trung bình".

In [ ]:
linear = LogisticRegression(max_iter=5000).fit(X_train, y_train)
phi_linear, _ = exact_shapley(linear.decision_function, customer, background)
closed_form = linear.coef_[0] * (customer.values[0] - background.mean().values)
print(f"sai lệch giữa Shapley chính xác và β·(x − E[x]): {np.abs(phi_linear.values - closed_form).max():.1e}")

## 3. Từ cục bộ đến toàn cục

Tính Shapley cho 150 khách hàng rồi tổng hợp: **mean |φ|** cho độ quan trọng toàn cục; biểu đồ φ theo giá trị đặc trưng cho thấy hình dạng tác động và mức độ khác nhau giữa các khách hàng (thay cho beeswarm/dependence plot của thư viện `shap`). Cài đặt vector hoá theo tập con để chạy nhanh hơn:

In [ ]:
def shapley_many(f, X_explain, background):
    """Shapley chính xác cho nhiều mẫu: với mỗi tập con S, tính v(S) cho mọi mẫu cùng lúc."""
    n, d = X_explain.shape
    cols = list(X_explain.columns)
    B = len(background)
    tiled_bg = pd.concat([background] * n, ignore_index=True)
    repeated_x = X_explain.loc[X_explain.index.repeat(B)].reset_index(drop=True)
    value = {}
    for size in range(d + 1):
        for S in itertools.combinations(range(d), size):
            Z = tiled_bg.copy()
            for j in S:
                Z[cols[j]] = repeated_x[cols[j]].values
            value[S] = f(Z).reshape(n, B).mean(1)
    phi = np.zeros((n, d))
    for j in range(d):
        for S, v_S in value.items():
            if j not in S:
                w = factorial(len(S)) * factorial(d - len(S) - 1) / factorial(d)
                phi[:, j] += w * (value[tuple(sorted(S + (j,)))] - v_S)
    return pd.DataFrame(phi, columns=cols, index=X_explain.index)


t0 = time.perf_counter()
X_explain = X_test.sample(150, random_state=1)
Phi = shapley_many(f, X_explain, X_train.sample(50, random_state=0))
print(f"{time.perf_counter() - t0:.0f}s | độ quan trọng toàn cục (mean |φ|):")
print(Phi.abs().mean().sort_values(ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, feat in zip(axes, ["tenure", "monthly", "support_calls", "total"]):
    ax.scatter(X_explain[feat], Phi[feat], s=10, c=X_explain["month_to_month"], cmap="coolwarm", alpha=0.8)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel(feat)
ax.figure.axes[0].set_ylabel("φ (log-odds)")
axes[0].set_title("màu: hợp đồng theo tháng (đỏ) / không (xanh)", fontsize=8)
plt.tight_layout()
plt.show()

Đóng góp của `tenure` giảm gần tuyến tính theo tenure, của `monthly` và `support_calls` tăng — đúng hướng và hình dạng của logit thật; `total` dao động quanh 0 với biên độ nhỏ. Thứ hạng mean |φ| gần với permutation importance ở bài 01 — nhưng hai đại lượng trả lời câu hỏi khác nhau: φ đo **mô hình dựa vào đặc trưng bao nhiêu khi đưa ra dự đoán**, permutation đo **hiệu năng phụ thuộc vào đặc trưng bao nhiêu**.

## 4. Counterfactual: "phải thay đổi gì?"

Khách hàng thường không hỏi "đóng góp Shapley của tôi bao nhiêu?" mà hỏi "**làm gì để kết quả khác đi?**". Một **giải thích phản thực tế** (Wachter et al., 2017) là thay đổi **nhỏ nhất, khả thi** đưa dự đoán qua ngưỡng. Chỉ thay đổi các đặc trưng **có thể hành động** (doanh nghiệp đề xuất được): loại hợp đồng, phương thức thanh toán, số sản phẩm, xử lý để giảm cuộc gọi hỗ trợ — không đổi tuổi (senior) hay thời gian gắn bó.

In [ ]:
p = model.predict_proba(X_test)[:, 1]
risky = X_test.iloc[[int(np.argmax(p))]]
print(f"khách hàng rủi ro cao nhất: xác suất rời bỏ = {p.max():.1%}")
print(risky[["month_to_month", "two_year", "ewallet", "n_products", "support_calls", "monthly"]])

options = {                                                            # (giá trị, "chi phí" thay đổi)
    "contract": [("giữ nguyên", {}, 0), ("1 năm", {"month_to_month": 0, "two_year": 0}, 2), ("2 năm", {"month_to_month": 0, "two_year": 1}, 3)],
    "payment": [("giữ nguyên", {}, 0), ("bỏ ví điện tử", {"ewallet": 0}, 1)],
    "products": [("giữ nguyên", {}, 0), ("+1 sản phẩm", {"n_products": min(risky.n_products.values[0] + 1, 4)}, 1)],
    "support": [("giữ nguyên", {}, 0), ("giảm 2 cuộc gọi", {"support_calls": max(risky.support_calls.values[0] - 2, 0)}, 2)],
}
candidates = []
for combo in itertools.product(*options.values()):
    changes = {k: v for _, change, _ in combo for k, v in change.items()}
    cost = sum(c for *_, c in combo)
    prob = model.predict_proba(risky.assign(**changes))[0, 1]
    candidates.append((cost, prob, " + ".join(name for name, _, c in combo if c > 0) or "không đổi gì"))
candidates.sort()
threshold = 0.5
feasible = [c for c in candidates if c[1] < threshold]
print(f"\ncounterfactual rẻ nhất đưa xác suất xuống dưới {threshold:.0%}:")
for cost, prob, desc in feasible[:3]:
    print(f"  chi phí {cost}: {desc:45s} → {prob:.1%}")

Counterfactual dễ hiểu và có thể hành động ("chuyển sang hợp đồng 2 năm và giải quyết các vấn đề khiến khách phải gọi hỗ trợ"), nhưng cần cẩn thận: (1) nó cho biết **mô hình** sẽ đổi dự đoán, không đảm bảo **khách hàng thật** sẽ ở lại — đó là câu hỏi nhân quả (chương 38: uplift); (2) phải ràng buộc tính khả thi (không đề xuất "trẻ lại 10 tuổi") và nhất quán (đổi hợp đồng thì giá có thể đổi theo); (3) có nhiều counterfactual khác nhau — chọn theo chi phí thực tế. Thư viện: DiCE, Alibi.

## 🧠 Kiểm tra nhanh

1. LIME tạo lời giải thích như thế nào? Những lựa chọn nào làm kết quả thay đổi?
2. Viết công thức giá trị Shapley. Bốn tính chất nào làm nó "công bằng"?
3. Hàm giá trị interventional là gì? Vì sao cần dữ liệu nền?
4. Shapley của mô hình tuyến tính có dạng gì?
5. Mean |φ| khác permutation importance thế nào?
6. Counterfactual là gì? Vì sao nó không đảm bảo tác động thật?

## 📌 Tóm tắt

- LIME: mô hình tuyến tính có trọng số quanh điểm cần giải thích — trực quan nhưng nhạy với số mẫu, kernel, cách nhiễu loạn.
- Shapley: chia $f(x) - \mathbb E[f]$ theo đóng góp biên trung bình qua mọi tập con; duy nhất thoả hiệu quả/đối xứng/dummy/cộng tính; cài chính xác khi ít đặc trưng, khớp `shap`; mô hình tuyến tính: $\beta_j(x_j - \mathbb E x_j)$.
- Tổng hợp SHAP cục bộ cho bức tranh toàn cục; counterfactual cho lời khuyên hành động nhưng không phải nhân quả.

## 📚 Tài liệu tham khảo

- Ribeiro, Singh & Guestrin (2016) — *"Why Should I Trust You?": Explaining the Predictions of Any Classifier* (LIME)
- Lundberg & Lee (2017) — *A Unified Approach to Interpreting Model Predictions* (SHAP)
- Lundberg et al. (2020) — *From local explanations to global understanding with explainable AI for trees* (TreeSHAP)
- Shapley (1953) — *A value for n-person games*
- Wachter, Mittelstadt & Russell (2017) — *Counterfactual Explanations without Opening the Black Box*
- Molnar — *Interpretable Machine Learning*, chương LIME, Shapley, SHAP, Counterfactual